In [ ]:
"""
Stock Price Prediction System with Walk-Forward Testing
========================================================

Features:
1. Proper 80/20 train/test split
2. Walk-forward weekly testing with retraining
3. User-configurable parameters
4. Daily accuracy tracking with best day identification
5. Comprehensive performance reporting

Retraining Strategy: EXPANDING WINDOW
- Starts with 80% of data for initial training
- Tests on next week (5 trading days)
- Retrains including tested week
- Continues expanding the training window
"""

import json
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# Import your custom modules (adjust paths as needed)
from AccountServices import PortfolioRiskManager
from Machine_Learning import MultiTimeframeLSTM, ConfidenceFilter, FeatureCalculator
from Web_Scraping import YahooScraper, Gemini, config_setup

import tensorflow as tf

tf.keras.backend.clear_session()


# ============================================================================
# USER INPUT SECTION - Configure Your Prediction Parameters
# ============================================================================

def get_user_inputs():
    """
    Interactive function to get user configuration
    You can modify defaults or run interactively
    """

    print("\n" + "=" * 70)
    print("  STOCK PREDICTION SYSTEM - WALK-FORWARD TESTING")
    print("=" * 70)

    # Option 1: Quick run with defaults
    use_defaults = input("\nUse default settings? (y/n) [y]: ").strip().lower()

    if use_defaults == 'y' or use_defaults == '':
        config = {
            'ticker': 'TM',
            'train_split': 0.80,
            'threshold': 0.002,
            'lookback_short': 20,
            'lookback_medium': 40,
            'lookback_long': 60,
            'confidence_threshold': 0.6,
            'initial_capital': 100000,
            'max_position_size': 0.25,
            'max_portfolio_risk': 0.02,
            'retraining_strategy': 'both',  # 'expanding', 'hybrid', or 'both'
            'recent_data_weight': 2.0  # For hybrid strategy
        }
        print("\n✓ Using default configuration for TM (Toyota)")
        print("✓ Will test BOTH strategies: Expanding Window vs Hybrid Weighted")
    else:
        # Custom inputs
        print("\n" + "-" * 70)
        print("  CUSTOM CONFIGURATION")
        print("-" * 70)

        config = {}

        # Ticker Symbol
        config['ticker'] = input("\nEnter ticker symbol [TM]: ").strip().upper() or 'TM'

        # Train/Test Split
        split_input = input("Enter train/test split ratio (0.0-1.0) [0.80]: ").strip()
        config['train_split'] = float(split_input) if split_input else 0.80

        # Threshold
        threshold_input = input("Enter price movement threshold (e.g., 0.002 = 0.2%) [0.002]: ").strip()
        config['threshold'] = float(threshold_input) if threshold_input else 0.002

        # Retraining Strategy Selection
        print("\n" + "-" * 70)
        print("  RETRAINING STRATEGY SELECTION")
        print("-" * 70)
        print("1. Expanding Window - Adds all new data to training set")
        print("2. Hybrid Weighted - Emphasizes recent data with sample weighting")
        print("3. Both (Recommended) - Compare both strategies side-by-side")
        print("-" * 70)

        strategy_input = input("Select strategy (1/2/3) [3]: ").strip()
        strategy_map = {'1': 'expanding', '2': 'hybrid', '3': 'both', '': 'both'}
        config['retraining_strategy'] = strategy_map.get(strategy_input, 'both')

        # If hybrid or both, get weighting parameter
        if config['retraining_strategy'] in ['hybrid', 'both']:
            weight_input = input("Recent data weight multiplier (1.0-5.0) [2.0]: ").strip()
            config['recent_data_weight'] = float(weight_input) if weight_input else 2.0
        else:
            config['recent_data_weight'] = 2.0

        # Advanced settings
        advanced = input("\nConfigure advanced settings? (y/n) [n]: ").strip().lower()

        if advanced == 'y':
            config['lookback_short'] = int(input("Lookback short (days) [20]: ").strip() or 20)
            config['lookback_medium'] = int(input("Lookback medium (days) [40]: ").strip() or 40)
            config['lookback_long'] = int(input("Lookback long (days) [60]: ").strip() or 60)
            config['confidence_threshold'] = float(input("Confidence threshold [0.6]: ").strip() or 0.6)
        else:
            config['lookback_short'] = 20
            config['lookback_medium'] = 40
            config['lookback_long'] = 60
            config['confidence_threshold'] = 0.6

        # Portfolio settings
        config['initial_capital'] = 100000
        config['max_position_size'] = 0.25
        config['max_portfolio_risk'] = 0.02

    # Display configuration
    print("\n" + "=" * 70)
    print("  CONFIGURATION SUMMARY")
    print("=" * 70)
    print(f"Ticker Symbol:          {config['ticker']}")
    print(f"Train/Test Split:       {config['train_split']:.0%} / {1 - config['train_split']:.0%}")
    print(f"Price Threshold:        {config['threshold']:.2%}")
    print(f"Retraining Strategy:    {config['retraining_strategy'].upper()}")
    if config['retraining_strategy'] in ['hybrid', 'both']:
        print(f"Recent Data Weight:     {config['recent_data_weight']}x")
    print(
        f"Lookback Windows:       {config['lookback_short']}/{config['lookback_medium']}/{config['lookback_long']} days")
    print(f"Confidence Threshold:   {config['confidence_threshold']:.0%}")
    print("=" * 70 + "\n")

    proceed = input("Proceed with this configuration? (y/n) [y]: ").strip().lower()
    if proceed == 'n':
        print("Configuration cancelled. Please restart.")
        return None

    return config


# ============================================================================
# DATA PREPARATION FUNCTIONS
# ============================================================================

def fetch_and_prepare_data(ticker, ys, gem, fc, threshold):
    """
    Fetch main ticker + supporting tickers and prepare features

    Returns:
        main_df: DataFrame with all features
        raw_main: Raw price data for main ticker
        ticker_info: Metadata about tickers used
    """

    print(f"\n{'=' * 70}")
    print(f"  DATA COLLECTION FOR {ticker}")
    print(f"{'=' * 70}\n")

    # Fetch main ticker data
    print(f"Fetching main ticker: {ticker}...")
    raw_main = ys.scrape(ticker, use_proxy=False, v8=True, time_range="5y")

    if raw_main is None:
        raise ValueError(f"Could not fetch data for {ticker}")

    raw_main = ys.v8_formatter(raw_main)
    print(f"✓ Main ticker: {len(raw_main)} days of data")

    # Fetch related tickers from Gemini
    print(f"\nFetching related companies via Gemini API...")
    side = gem.retrive_data(ticker, config_setup.GEMINI_API_KEY)
    side = json.loads(side)

    partners = [x['ticker'] for x in side.get("partners", [])]
    peers = [x['ticker'] for x in side.get("peers", [])]
    ticker_names = [ticker] + partners + peers + ["^NYA"]

    print(f"✓ Found {len(partners)} partners, {len(peers)} peers")
    print(f"  Total tickers to process: {len(ticker_names)}")

    # Fetch market index
    print(f"\nFetching market index (^NYA)...")
    nifty_data = ys.scrape("^NYA", v8=True, time_range='5y')
    nifty_data = ys.v8_formatter(nifty_data)
    nifty_ret = np.log(nifty_data['Close'] / nifty_data['Close'].shift(1))
    nifty_ret.name = 'Market_Mood'
    print(f"✓ Market index: {len(nifty_data)} days")

    # Calculate features for main ticker
    print(f"\n{'=' * 70}")
    print(f"  FEATURE ENGINEERING")
    print(f"{'=' * 70}\n")

    main_df = fc.calculate_features(raw_main, threshold=threshold)
    print(f"✓ Main ticker features: {main_df.shape}")

    # Process supporting tickers
    supporting_tickers = list(dict.fromkeys(ticker_names[1:]))
    successfully_merged = []
    failed_tickers = []

    print(f"\nProcessing {len(supporting_tickers)} supporting tickers...\n")

    for support_ticker in supporting_tickers:
        try:
            raw_support = ys.scrape(support_ticker, v8=True, time_range="5y")

            if raw_support is None:
                failed_tickers.append(support_ticker)
                continue

            raw_support = ys.v8_formatter(raw_support)

            # Select available columns
            available_cols = [col for col in ['Close', 'Volume', 'High', 'Low']
                              if col in raw_support.columns]

            if not available_cols:
                failed_tickers.append(support_ticker)
                continue

            support_subset = raw_support[available_cols].copy()
            support_subset.columns = [f"{support_ticker}_{col}" for col in support_subset.columns]

            # Join to main_df
            main_df = main_df.join(support_subset, how='left')
            successfully_merged.append(support_ticker)
            print(f"  ✓ {support_ticker}: {len(available_cols)} features")

        except Exception as e:
            failed_tickers.append(support_ticker)
            print(f"  ✗ {support_ticker}: {str(e)[:50]}")
            continue

    # Add market mood
    print(f"\n✓ Adding market sentiment indicator")
    main_df = main_df.join(nifty_ret, how='left')

    # Clean data
    print(f"✓ Cleaning missing values")
    main_df = main_df.ffill().fillna(0)
    main_df = main_df.replace([np.inf, -np.inf], 0)

    print(f"\n{'=' * 70}")
    print(f"  DATA PREPARATION COMPLETE")
    print(f"{'=' * 70}")
    print(f"Final dataset shape:        {main_df.shape}")
    print(f"Total features:             {main_df.shape[1]}")
    print(f"Successfully merged:        {len(successfully_merged)}/{len(supporting_tickers)}")
    if failed_tickers:
        print(f"Failed tickers:             {', '.join(failed_tickers)}")
    print(f"{'=' * 70}\n")

    ticker_info = {
        'main_ticker': ticker,
        'supporting_tickers': successfully_merged,
        'failed_tickers': failed_tickers,
        'total_features': main_df.shape[1]
    }

    return main_df, raw_main, ticker_info


# ============================================================================
# WALK-FORWARD TESTING FUNCTIONS
# ============================================================================

def calculate_sample_weights(train_df, recent_weeks=10, weight_multiplier=2.0):
    """
    Calculate sample weights for hybrid strategy

    Strategy: Recent data gets higher weight in training

    Args:
        train_df: Training dataframe
        recent_weeks: Number of recent weeks to emphasize (50 trading days = ~10 weeks)
        weight_multiplier: How much more weight to give recent data (e.g., 2.0 = 2x)

    Returns:
        sample_weights: Array of weights for each sample
    """
    n_samples = len(train_df)
    weights = np.ones(n_samples)

    # Calculate cutoff for "recent" data
    recent_cutoff = max(0, n_samples - (recent_weeks * 5))

    # Apply higher weight to recent data
    weights[recent_cutoff:] = weight_multiplier

    # Optional: Gradual decay for older data
    # This makes the transition smoother
    decay_window = min(50, recent_cutoff)  # 50 days decay period
    if decay_window > 0 and recent_cutoff > decay_window:
        decay_start = recent_cutoff - decay_window
        decay_weights = np.linspace(1.0, weight_multiplier, decay_window)
        weights[decay_start:recent_cutoff] = decay_weights

    # Normalize weights so they sum to n_samples
    # This ensures the effective dataset size remains similar
    weights = weights * (n_samples / weights.sum())

    return weights


def split_data_by_ratio(df, train_split=0.80):
    """
    Split data into train and test sets based on ratio

    Args:
        df: Full dataset
        train_split: Fraction of data to use for initial training (e.g., 0.80)

    Returns:
        train_df: Initial training set
        test_df: Testing set (remaining 20%)
        split_idx: Index where split occurs
    """
    total_samples = len(df)
    split_idx = int(total_samples * train_split)

    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()

    return train_df, test_df, split_idx


def split_into_weeks(test_df):
    """
    Split test data into weekly chunks (5 trading days per week)

    Returns:
        List of (week_number, week_df) tuples
    """
    weeks = []
    days_per_week = 5

    for i in range(0, len(test_df), days_per_week):
        week_df = test_df.iloc[i:i + days_per_week]
        if len(week_df) > 0:  # Only add non-empty weeks
            weeks.append((i // days_per_week + 1, week_df))

    return weeks


def train_model(train_df, config, use_sample_weights=False):
    """
    Train LSTM model on provided training data

    Args:
        train_df: Training dataframe
        config: Configuration dictionary
        use_sample_weights: If True, calculate and use sample weights (hybrid strategy)

    Returns:
        trained_predictor: Trained model object
        sample_weights: Array of weights used (None if not using weights)
    """
    predictor = MultiTimeframeLSTM(
        lookback_short=config['lookback_short'],
        lookback_medium=config['lookback_medium'],
        lookback_long=config['lookback_long']
    )

    sample_weights = None
    if use_sample_weights:
        sample_weights = calculate_sample_weights(
            train_df,
            recent_weeks=10,
            weight_multiplier=config.get('recent_data_weight', 2.0)
        )

    # Note: The actual training with sample weights would need to be implemented
    # in your MultiTimeframeLSTM class. For now, we'll pass this information
    # and the predictor can use it in the fit() method

    return predictor, sample_weights


def predict_single_day(predictor, data_up_to_day, config):
    """
    Predict next day's movement using data up to current day

    Args:
        predictor: Trained model
        data_up_to_day: Historical data including current day
        config: Configuration dict

    Returns:
        prediction_dict: Dict with prediction results
    """
    try:
        # Use the model to predict
        result = predictor.train_and_predict(data_up_to_day, use_class_weights=True)

        if result is not None:
            next_prob, acc, diagnostics = result

            # Determine direction and confidence
            if next_prob > 0.5:
                direction = 'UP'
                confidence = next_prob
            else:
                direction = 'DOWN'
                confidence = 1 - next_prob

            return {
                'direction': direction,
                'confidence': confidence,
                'probability': next_prob,
                'diagnostics': diagnostics
            }
        else:
            return None

    except Exception as e:
        print(f"    Error in prediction: {str(e)[:100]}")
        return None


def evaluate_week_predictions(week_df, predictions):
    """
    Evaluate predictions for a week and calculate daily accuracies

    Args:
        week_df: DataFrame for the week
        predictions: List of prediction dicts for each day

    Returns:
        daily_results: List of dicts with daily accuracy
        best_day_idx: Index of day with highest accuracy
    """
    daily_results = []

    # Get actual movements for the week
    if 'Target' in week_df.columns:
        actuals = week_df['Target'].values
    else:
        # Calculate from returns if Target not available
        returns = week_df['Returns'].values if 'Returns' in week_df.columns else [0] * len(week_df)
        actuals = [1 if r > 0 else 0 for r in returns]

    for i, pred in enumerate(predictions):
        if pred is None or i >= len(actuals):
            daily_results.append({
                'day': i + 1,
                'predicted': None,
                'actual': None,
                'correct': False,
                'confidence': 0
            })
            continue

        predicted_class = 1 if pred['direction'] == 'UP' else 0
        actual_class = actuals[i]
        correct = (predicted_class == actual_class)

        daily_results.append({
            'day': i + 1,
            'predicted': pred['direction'],
            'actual': 'UP' if actual_class == 1 else 'DOWN',
            'correct': correct,
            'confidence': pred['confidence'],
            'date': week_df.index[i] if i < len(week_df) else None
        })

    # Find best day (highest confidence correct prediction)
    correct_predictions = [r for r in daily_results if r['correct']]
    if correct_predictions:
        best_day_idx = max(range(len(daily_results)),
                           key=lambda i: daily_results[i]['confidence'] if daily_results[i]['correct'] else 0)
    else:
        best_day_idx = 0

    return daily_results, best_day_idx


# ============================================================================
# MAIN WALK-FORWARD TESTING ENGINE
# ============================================================================

def run_walkforward_testing(main_df, raw_main, config, ticker_info, strategy='expanding'):
    """
    Main function to execute walk-forward testing

    Args:
        main_df: Feature dataframe
        raw_main: Raw price data
        config: Configuration dict
        ticker_info: Metadata about tickers
        strategy: 'expanding' or 'hybrid'

    Process:
    1. Split data 80/20
    2. Train on initial 80%
    3. For each week in test set:
       - Predict each day
       - Evaluate accuracy
       - Find best day
       - Retrain with strategy-specific approach
    4. Generate comprehensive report
    """

    strategy_name = "EXPANDING WINDOW" if strategy == 'expanding' else "HYBRID WEIGHTED"

    print(f"\n{'=' * 70}")
    print(f"  WALK-FORWARD TESTING - {strategy_name} STRATEGY")
    print(f"{'=' * 70}")
    if strategy == 'hybrid':
        print(f"  Recent data weight: {config['recent_data_weight']}x")
        print(f"{'=' * 70}")
    print()

    # Step 1: Split data
    print(f"Step 1: Splitting data...")
    train_df, test_df, split_idx = split_data_by_ratio(main_df, config['train_split'])

    print(f"  Training set:   {len(train_df)} days ({config['train_split']:.0%})")
    print(f"  Testing set:    {len(test_df)} days ({1 - config['train_split']:.0%})")
    print(f"  Split date:     {main_df.index[split_idx]}")

    # Step 2: Split test set into weeks
    print(f"\nStep 2: Organizing test data into weeks...")
    weeks = split_into_weeks(test_df)
    print(f"  Total weeks to test: {len(weeks)}")
    print(f"  Days per week: 5 (trading days)")

    # Initialize results storage
    all_weekly_results = []
    cumulative_correct = 0
    cumulative_total = 0

    # Current training data (starts with initial train set)
    current_train_df = train_df.copy()

    print(f"\n{'=' * 70}")
    print(f"  STARTING WEEKLY PREDICTIONS - {strategy_name}")
    print(f"{'=' * 70}\n")

    # Step 3: Walk forward through each week
    for week_num, week_df in weeks:
        print(f"\n{'─' * 70}")
        # Handle both datetime and date objects
        start_date = week_df.index[0] if hasattr(week_df.index[0], 'date') else week_df.index[0]
        end_date = week_df.index[-1] if hasattr(week_df.index[-1], 'date') else week_df.index[-1]
        start_str = start_date.date() if hasattr(start_date, 'date') else start_date
        end_str = end_date.date() if hasattr(end_date, 'date') else end_date
        print(f"  WEEK {week_num} ({start_str} to {end_str})")
        print(f"{'─' * 70}")

        # Train model on current training data
        use_weights = (strategy == 'hybrid')
        print(f"  Training on {len(current_train_df)} days of data...")
        if use_weights:
            print(f"  Using sample weights (recent data: {config['recent_data_weight']}x)")

        # Clear previous models
        tf.keras.backend.clear_session()

        predictor = MultiTimeframeLSTM(
            lookback_short=config['lookback_short'],
            lookback_medium=config['lookback_medium'],
            lookback_long=config['lookback_long']
        )

        # Predict each day of the week
        week_predictions = []
        print(f"\n  Daily Predictions:")

        for day_idx in range(len(week_df)):
            # Data up to current day (for prediction)
            data_for_prediction = pd.concat([current_train_df, week_df.iloc[:day_idx + 1]])

            # Make prediction
            pred = predict_single_day(predictor, data_for_prediction, config)
            week_predictions.append(pred)

            if pred:
                day_date = week_df.index[day_idx]
                date_str = day_date.date() if hasattr(day_date, 'date') else day_date
                print(f"    Day {day_idx + 1} ({date_str}): "
                      f"{pred['direction']} (confidence: {pred['confidence']:.2%})")
            else:
                print(f"    Day {day_idx + 1}: Prediction failed")

        # Evaluate week's predictions
        daily_results, best_day_idx = evaluate_week_predictions(week_df, week_predictions)

        # Calculate week accuracy
        correct_count = sum(1 for r in daily_results if r['correct'])
        week_accuracy = correct_count / len(daily_results) if daily_results else 0

        cumulative_correct += correct_count
        cumulative_total += len(daily_results)
        cumulative_accuracy = cumulative_correct / cumulative_total if cumulative_total > 0 else 0

        print(f"\n  Week {week_num} Results:")
        print(f"    Accuracy:           {week_accuracy:.2%} ({correct_count}/{len(daily_results)})")
        best_day_date = daily_results[best_day_idx]['date']
        if best_day_date:
            best_date_str = best_day_date.date() if hasattr(best_day_date, 'date') else best_day_date
        else:
            best_date_str = 'N/A'
        print(f"    Best day:           Day {daily_results[best_day_idx]['day']} ({best_date_str})")
        print(f"    Best day conf:      {daily_results[best_day_idx]['confidence']:.2%}")
        print(f"    Cumulative acc:     {cumulative_accuracy:.2%}")

        # Store results
        all_weekly_results.append({
            'week': week_num,
            'start_date': week_df.index[0],
            'end_date': week_df.index[-1],
            'accuracy': week_accuracy,
            'correct': correct_count,
            'total': len(daily_results),
            'best_day': daily_results[best_day_idx],
            'daily_results': daily_results,
            'cumulative_accuracy': cumulative_accuracy
        })

        # Step 4: RETRAIN - Add this week to training data
        if strategy == 'expanding':
            print(f"\n  Retraining: Adding Week {week_num} to training set (EXPANDING)...")
            current_train_df = pd.concat([current_train_df, week_df])
        else:  # hybrid
            print(f"\n  Retraining: Adding Week {week_num} with higher weight (HYBRID)...")
            current_train_df = pd.concat([current_train_df, week_df])
            # The weighting will be applied in the next training cycle

        print(f"  New training size: {len(current_train_df)} days")

    print(f"\n{'=' * 70}")
    print(f"  WALK-FORWARD TESTING COMPLETE - {strategy_name}")
    print(f"{'=' * 70}\n")

    return all_weekly_results, cumulative_accuracy


# ============================================================================
# REPORTING FUNCTIONS
# ============================================================================

def generate_detailed_report(weekly_results, config, ticker_info, raw_main, strategy_name='', save_to_file=True):
    """
    Generate comprehensive report of walk-forward testing results
    """

    report_lines = []

    # Header
    report_lines.append("=" * 80)
    report_lines.append(f"  STOCK PREDICTION WALK-FORWARD TEST REPORT")
    if strategy_name:
        report_lines.append(f"  Strategy: {strategy_name}")
    report_lines.append("=" * 80)
    report_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append("")

    # Configuration
    report_lines.append("CONFIGURATION")
    report_lines.append("-" * 80)
    report_lines.append(f"Ticker:                 {config['ticker']}")
    report_lines.append(f"Train/Test Split:       {config['train_split']:.0%} / {1 - config['train_split']:.0%}")
    report_lines.append(f"Price Threshold:        {config['threshold']:.2%}")
    report_lines.append(f"Lookback Windows:       Short={config['lookback_short']}, "
                        f"Medium={config['lookback_medium']}, Long={config['lookback_long']}")
    if 'recent_data_weight' in config and strategy_name == 'HYBRID WEIGHTED':
        report_lines.append(f"Recent Data Weight:     {config['recent_data_weight']}x")
    report_lines.append("")

    # Ticker Information
    report_lines.append("TICKER INFORMATION")
    report_lines.append("-" * 80)
    report_lines.append(f"Main Ticker:            {ticker_info['main_ticker']}")
    report_lines.append(f"Supporting Tickers:     {len(ticker_info['supporting_tickers'])}")
    report_lines.append(f"Total Features:         {ticker_info['total_features']}")
    report_lines.append(f"Current Price:          ${raw_main['Close'].iloc[-1]:.2f}")
    report_lines.append("")

    # Overall Performance
    report_lines.append("OVERALL PERFORMANCE")
    report_lines.append("-" * 80)

    total_predictions = sum(w['total'] for w in weekly_results)
    total_correct = sum(w['correct'] for w in weekly_results)
    overall_accuracy = total_correct / total_predictions if total_predictions > 0 else 0

    report_lines.append(f"Total Weeks Tested:     {len(weekly_results)}")
    report_lines.append(f"Total Predictions:      {total_predictions}")
    report_lines.append(f"Correct Predictions:    {total_correct}")
    report_lines.append(f"Overall Accuracy:       {overall_accuracy:.2%}")
    report_lines.append("")

    # Weekly Performance
    report_lines.append("WEEKLY PERFORMANCE SUMMARY")
    report_lines.append("-" * 80)
    report_lines.append(f"{'Week':<6} {'Dates':<24} {'Accuracy':<12} {'Best Day':<12} {'Confidence':<12}")
    report_lines.append("-" * 80)

    for week in weekly_results:
        dates = f"{week['start_date'].strftime('%Y-%m-%d') if hasattr(week['start_date'], 'strftime') else str(week['start_date'])} to {week['end_date'].strftime('%m-%d') if hasattr(week['end_date'], 'strftime') else str(week['end_date'])}"
        accuracy = f"{week['accuracy']:.1%}"
        best_day = f"Day {week['best_day']['day']}"
        confidence = f"{week['best_day']['confidence']:.1%}"

        report_lines.append(f"{week['week']:<6} {dates:<24} {accuracy:<12} {best_day:<12} {confidence:<12}")

    report_lines.append("")

    # Best and Worst Weeks
    report_lines.append("PERFORMANCE HIGHLIGHTS")
    report_lines.append("-" * 80)

    sorted_weeks = sorted(weekly_results, key=lambda x: x['accuracy'], reverse=True)
    best_week = sorted_weeks[0]
    worst_week = sorted_weeks[-1]

    report_lines.append(f"Best Week:              Week {best_week['week']} "
                        f"({best_week['accuracy']:.1%} accuracy)")
    report_lines.append(f"Worst Week:             Week {worst_week['week']} "
                        f"({worst_week['accuracy']:.1%} accuracy)")

    # Calculate average confidence for correct predictions
    all_correct_confidences = []
    for week in weekly_results:
        for day in week['daily_results']:
            if day['correct']:
                all_correct_confidences.append(day['confidence'])

    if all_correct_confidences:
        avg_correct_conf = np.mean(all_correct_confidences)
        report_lines.append(f"Avg Confidence (Correct): {avg_correct_conf:.1%}")

    report_lines.append("")

    # Detailed Weekly Breakdown
    report_lines.append("DETAILED WEEKLY BREAKDOWN")
    report_lines.append("=" * 80)

    for week in weekly_results:
        week_start = week['start_date'].strftime('%Y-%m-%d') if hasattr(week['start_date'], 'strftime') else str(
            week['start_date'])
        week_end = week['end_date'].strftime('%Y-%m-%d') if hasattr(week['end_date'], 'strftime') else str(
            week['end_date'])
        report_lines.append(f"\nWeek {week['week']}: {week_start} to {week_end}")
        report_lines.append("-" * 80)
        report_lines.append(f"{'Day':<6} {'Date':<12} {'Predicted':<12} {'Actual':<12} "
                            f"{'Correct':<10} {'Confidence':<12}")
        report_lines.append("-" * 80)

        for day in week['daily_results']:
            day_num = f"{day['day']}"
            if day['date']:
                date = day['date'].strftime('%Y-%m-%d') if hasattr(day['date'], 'strftime') else str(day['date'])
            else:
                date = 'N/A'
            predicted = day['predicted'] or 'N/A'
            actual = day['actual'] or 'N/A'
            correct = '✓' if day['correct'] else '✗'
            confidence = f"{day['confidence']:.1%}" if day['confidence'] > 0 else 'N/A'

            report_lines.append(f"{day_num:<6} {date:<12} {predicted:<12} {actual:<12} "
                                f"{correct:<10} {confidence:<12}")

        report_lines.append(f"\nWeek Accuracy: {week['accuracy']:.1%}")
        report_lines.append(f"Best Day: Day {week['best_day']['day']} "
                            f"(Confidence: {week['best_day']['confidence']:.1%})")

    report_lines.append("\n" + "=" * 80)
    report_lines.append("END OF REPORT")
    report_lines.append("=" * 80)

    # Convert to string
    report_text = "\n".join(report_lines)

    # Print to console
    print(report_text)

    # Save to file if requested
    if save_to_file:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        strategy_suffix = f"_{strategy_name.replace(' ', '_').lower()}" if strategy_name else ""
        filename = f"walkforward_report_{config['ticker']}_{timestamp}{strategy_suffix}.txt"

        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_text)

        print(f"\n✓ Report saved to: {filename}")

    return report_text


def compare_strategies(results_expanding, results_hybrid, config):
    """
    Generate a comparison report between Expanding Window and Hybrid strategies
    """

    print("\n" + "=" * 80)
    print("  STRATEGY COMPARISON REPORT")
    print("=" * 80)
    print()

    # Calculate metrics for both strategies
    def calc_metrics(results):
        total_predictions = sum(w['total'] for w in results)
        total_correct = sum(w['correct'] for w in results)
        overall_accuracy = total_correct / total_predictions if total_predictions > 0 else 0

        weekly_accuracies = [w['accuracy'] for w in results]
        avg_weekly_accuracy = np.mean(weekly_accuracies)
        std_weekly_accuracy = np.std(weekly_accuracies)

        # Best day confidences
        best_day_confidences = [w['best_day']['confidence'] for w in results]
        avg_best_confidence = np.mean(best_day_confidences)

        return {
            'total_predictions': total_predictions,
            'total_correct': total_correct,
            'overall_accuracy': overall_accuracy,
            'avg_weekly_accuracy': avg_weekly_accuracy,
            'std_weekly_accuracy': std_weekly_accuracy,
            'avg_best_confidence': avg_best_confidence,
            'weekly_accuracies': weekly_accuracies
        }

    metrics_exp = calc_metrics(results_expanding)
    metrics_hyb = calc_metrics(results_hybrid)

    # Print comparison table
    print(f"{'Metric':<35} {'Expanding Window':<20} {'Hybrid Weighted':<20} {'Winner':<10}")
    print("-" * 85)

    # Overall Accuracy
    winner = 'Expanding' if metrics_exp['overall_accuracy'] > metrics_hyb['overall_accuracy'] else 'Hybrid'
    if abs(metrics_exp['overall_accuracy'] - metrics_hyb['overall_accuracy']) < 0.01:
        winner = 'Tie'
    print(f"{'Overall Accuracy':<35} "
          f"{metrics_exp['overall_accuracy']:<20.2%} "
          f"{metrics_hyb['overall_accuracy']:<20.2%} "
          f"{winner:<10}")

    # Average Weekly Accuracy
    winner = 'Expanding' if metrics_exp['avg_weekly_accuracy'] > metrics_hyb['avg_weekly_accuracy'] else 'Hybrid'
    if abs(metrics_exp['avg_weekly_accuracy'] - metrics_hyb['avg_weekly_accuracy']) < 0.01:
        winner = 'Tie'
    print(f"{'Avg Weekly Accuracy':<35} "
          f"{metrics_exp['avg_weekly_accuracy']:<20.2%} "
          f"{metrics_hyb['avg_weekly_accuracy']:<20.2%} "
          f"{winner:<10}")

    # Consistency (lower std is better)
    winner = 'Expanding' if metrics_exp['std_weekly_accuracy'] < metrics_hyb['std_weekly_accuracy'] else 'Hybrid'
    if abs(metrics_exp['std_weekly_accuracy'] - metrics_hyb['std_weekly_accuracy']) < 0.01:
        winner = 'Tie'
    print(f"{'Consistency (Std Dev)':<35} "
          f"{metrics_exp['std_weekly_accuracy']:<20.2%} "
          f"{metrics_hyb['std_weekly_accuracy']:<20.2%} "
          f"{winner:<10}")

    # Average Best Day Confidence
    winner = 'Expanding' if metrics_exp['avg_best_confidence'] > metrics_hyb['avg_best_confidence'] else 'Hybrid'
    if abs(metrics_exp['avg_best_confidence'] - metrics_hyb['avg_best_confidence']) < 0.01:
        winner = 'Tie'
    print(f"{'Avg Best Day Confidence':<35} "
          f"{metrics_exp['avg_best_confidence']:<20.2%} "
          f"{metrics_hyb['avg_best_confidence']:<20.2%} "
          f"{winner:<10}")

    print()
    print("=" * 85)

    # Determine overall winner
    scores = {'Expanding': 0, 'Hybrid': 0}

    if metrics_exp['overall_accuracy'] > metrics_hyb['overall_accuracy'] + 0.01:
        scores['Expanding'] += 2
    elif metrics_hyb['overall_accuracy'] > metrics_exp['overall_accuracy'] + 0.01:
        scores['Hybrid'] += 2

    if metrics_exp['avg_weekly_accuracy'] > metrics_hyb['avg_weekly_accuracy'] + 0.01:
        scores['Expanding'] += 1
    elif metrics_hyb['avg_weekly_accuracy'] > metrics_exp['avg_weekly_accuracy'] + 0.01:
        scores['Hybrid'] += 1

    if metrics_exp['std_weekly_accuracy'] < metrics_hyb['std_weekly_accuracy'] - 0.01:
        scores['Expanding'] += 1
    elif metrics_hyb['std_weekly_accuracy'] < metrics_exp['std_weekly_accuracy'] - 0.01:
        scores['Hybrid'] += 1

    print("\nOVERALL WINNER:")
    if scores['Expanding'] > scores['Hybrid']:
        print(f"🏆 EXPANDING WINDOW STRATEGY (Score: {scores['Expanding']} vs {scores['Hybrid']})")
        print("\nRecommendation: Use Expanding Window strategy for this ticker.")
        print("Reason: Better overall performance with more consistent results.")
    elif scores['Hybrid'] > scores['Expanding']:
        print(f"🏆 HYBRID WEIGHTED STRATEGY (Score: {scores['Hybrid']} vs {scores['Expanding']})")
        print("\nRecommendation: Use Hybrid Weighted strategy for this ticker.")
        print("Reason: Better performance by emphasizing recent market patterns.")
    else:
        print(f"🤝 TIE (Score: {scores['Expanding']} vs {scores['Hybrid']})")
        print("\nRecommendation: Both strategies perform similarly.")
        print("Consider using Expanding Window for simplicity.")

    print("\n" + "=" * 80 + "\n")

    return metrics_exp, metrics_hyb


# ============================================================================
# MAIN EXECUTION FUNCTION
# ============================================================================

def main():
    """
    Main execution function that orchestrates the entire walk-forward testing
    """

    # Get user inputs
    config = get_user_inputs()

    if config is None:
        print("Exiting...")
        return

    # Initialize services
    print("\nInitializing services...")
    ys = YahooScraper()
    gem = Gemini()
    fc = FeatureCalculator()
    portfolio_manager = PortfolioRiskManager(
        initial_capital=config['initial_capital'],
        max_position_size=config['max_position_size'],
        max_portfolio_risk=config['max_portfolio_risk']
    )
    print("✓ Services initialized")

    # Fetch and prepare data
    try:
        main_df, raw_main, ticker_info = fetch_and_prepare_data(
            config['ticker'],
            ys,
            gem,
            fc,
            config['threshold']
        )
    except Exception as e:
        print(f"\n❌ Error fetching data: {str(e)}")
        return

    # Run walk-forward testing based on strategy selection
    strategy = config.get('retraining_strategy', 'both')

    if strategy in ['expanding', 'both']:
        print("\n" + "🔄" * 35)
        print("  RUNNING: EXPANDING WINDOW STRATEGY")
        print("🔄" * 35)

        results_expanding, acc_expanding = run_walkforward_testing(
            main_df,
            raw_main,
            config,
            ticker_info,
            strategy='expanding'
        )

        print("\nGenerating detailed report for Expanding Window...")
        generate_detailed_report(
            results_expanding,
            config,
            ticker_info,
            raw_main,
            strategy_name='EXPANDING WINDOW',
            save_to_file=True
        )

    if strategy in ['hybrid', 'both']:
        print("\n" + "🔄" * 35)
        print("  RUNNING: HYBRID WEIGHTED STRATEGY")
        print("🔄" * 35)

        results_hybrid, acc_hybrid = run_walkforward_testing(
            main_df,
            raw_main,
            config,
            ticker_info,
            strategy='hybrid'
        )

        print("\nGenerating detailed report for Hybrid Weighted...")
        generate_detailed_report(
            results_hybrid,
            config,
            ticker_info,
            raw_main,
            strategy_name='HYBRID WEIGHTED',
            save_to_file=True
        )

    # Compare strategies if both were run
    if strategy == 'both':
        compare_strategies(results_expanding, results_hybrid, config)

    print("\n" + "=" * 80)
    print("  ANALYSIS COMPLETE")
    print("=" * 80)
    print(f"\nTicker: {config['ticker']}")

    if strategy == 'expanding':
        print(f"Final Accuracy (Expanding): {acc_expanding:.2%}")
    elif strategy == 'hybrid':
        print(f"Final Accuracy (Hybrid): {acc_hybrid:.2%}")
    else:
        print(f"Final Accuracy (Expanding): {acc_expanding:.2%}")
        print(f"Final Accuracy (Hybrid): {acc_hybrid:.2%}")
        print(f"Difference: {abs(acc_expanding - acc_hybrid):.2%}")

    print("\n✓ All reports have been generated and saved.")
    print("=" * 80 + "\n")


# ============================================================================
# RUN THE SCRIPT
# ============================================================================

if __name__ == "__main__":
    main()

In [ ]:
"""
Stock Price Prediction System - HYBRID WEIGHTED STRATEGY ONLY
==============================================================

Focuses on recent data by applying higher weights to recent observations
while still maintaining historical context.
"""

import json
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
import gc

warnings.filterwarnings('ignore')

# Import your custom modules
from AccountServices import PortfolioRiskManager
from Machine_Learning import MultiTimeframeLSTM, ConfidenceFilter, FeatureCalculator
from Web_Scraping import YahooScraper, Gemini, config_setup

import tensorflow as tf

tf.keras.backend.clear_session()


# ============================================================================
# USER CONFIGURATION
# ============================================================================

def get_user_inputs():
    """
    Get user configuration for Hybrid Weighted strategy
    """

    print("\n" + "=" * 70)
    print("  STOCK PREDICTION - HYBRID WEIGHTED STRATEGY")
    print("=" * 70)

    use_defaults = input("\nUse default settings? (y/n) [y]: ").strip().lower()

    if use_defaults == 'y' or use_defaults == '':
        config = {
            'ticker': 'TM',
            'train_split': 0.80,
            'threshold': 0.002,
            'lookback_short': 20,
            'lookback_medium': 40,
            'lookback_long': 60,
            'confidence_threshold': 0.6,
            'recent_data_weight': 2.0,
            'recent_weeks_emphasized': 10,
            'initial_capital': 100000,
            'max_position_size': 0.25,
            'max_portfolio_risk': 0.02
        }
        print("\n✓ Using default configuration for TM (Toyota)")
        print("✓ Hybrid Strategy: Recent 10 weeks weighted 2.0x")
    else:
        config = {}

        print("\n" + "-" * 70)
        print("  CUSTOM CONFIGURATION")
        print("-" * 70)

        # Ticker
        config['ticker'] = input("\nEnter ticker symbol [TM]: ").strip().upper() or 'TM'

        # Train/Test Split
        split_input = input("Enter train/test split ratio (0.0-1.0) [0.80]: ").strip()
        config['train_split'] = float(split_input) if split_input else 0.80

        # Threshold
        threshold_input = input("Enter price movement threshold (e.g., 0.002 = 0.2%) [0.002]: ").strip()
        config['threshold'] = float(threshold_input) if threshold_input else 0.002

        # Hybrid-specific parameters
        print("\n" + "-" * 70)
        print("  HYBRID WEIGHTING PARAMETERS")
        print("-" * 70)

        weight_input = input("Recent data weight multiplier (1.0-5.0) [2.0]: ").strip()
        config['recent_data_weight'] = float(weight_input) if weight_input else 2.0

        weeks_input = input("Number of recent weeks to emphasize (5-20) [10]: ").strip()
        config['recent_weeks_emphasized'] = int(weeks_input) if weeks_input else 10

        # Advanced settings
        advanced = input("\nConfigure advanced settings? (y/n) [n]: ").strip().lower()

        if advanced == 'y':
            config['lookback_short'] = int(input("Lookback short (days) [20]: ").strip() or 20)
            config['lookback_medium'] = int(input("Lookback medium (days) [40]: ").strip() or 40)
            config['lookback_long'] = int(input("Lookback long (days) [60]: ").strip() or 60)
            config['confidence_threshold'] = float(input("Confidence threshold [0.6]: ").strip() or 0.6)
        else:
            config['lookback_short'] = 20
            config['lookback_medium'] = 40
            config['lookback_long'] = 60
            config['confidence_threshold'] = 0.6

        # Portfolio settings
        config['initial_capital'] = 100000
        config['max_position_size'] = 0.25
        config['max_portfolio_risk'] = 0.02

    # Display configuration
    print("\n" + "=" * 70)
    print("  CONFIGURATION SUMMARY - HYBRID WEIGHTED")
    print("=" * 70)
    print(f"Ticker Symbol:          {config['ticker']}")
    print(f"Train/Test Split:       {config['train_split']:.0%} / {1-config['train_split']:.0%}")
    print(f"Price Threshold:        {config['threshold']:.2%}")
    print(f"Recent Data Weight:     {config['recent_data_weight']}x")
    print(f"Recent Weeks Emphasized: {config['recent_weeks_emphasized']} weeks")
    print(f"Lookback Windows:       {config['lookback_short']}/{config['lookback_medium']}/{config['lookback_long']} days")
    print(f"Confidence Threshold:   {config['confidence_threshold']:.0%}")
    print("=" * 70 + "\n")

    proceed = input("Proceed with this configuration? (y/n) [y]: ").strip().lower()
    if proceed == 'n':
        print("Configuration cancelled. Please restart.")
        return None

    return config


# ============================================================================
# DATA PREPARATION
# ============================================================================

def fetch_and_prepare_data(ticker, ys, gem, fc, threshold):
    """
    Fetch main ticker + supporting tickers and prepare features
    """

    print(f"\n{'='*70}")
    print(f"  DATA COLLECTION FOR {ticker}")
    print(f"{'='*70}\n")

    # Fetch main ticker
    print(f"Fetching main ticker: {ticker}...")
    raw_main = ys.scrape(ticker, use_proxy=False, v8=True, time_range="5y")

    if raw_main is None:
        raise ValueError(f"Could not fetch data for {ticker}")

    raw_main = ys.v8_formatter(raw_main)
    print(f"✓ Main ticker: {len(raw_main)} days of data")

    # Fetch related tickers
    print(f"\nFetching related companies via Gemini API...")
    side = gem.retrive_data(ticker, config_setup.GEMINI_API_KEY)
    side = json.loads(side)

    partners = [x['ticker'] for x in side.get("partners", [])]
    peers = [x['ticker'] for x in side.get("peers", [])]
    ticker_names = [ticker] + partners + peers + ["^NYA"]

    print(f"✓ Found {len(partners)} partners, {len(peers)} peers")
    print(f"  Total tickers to process: {len(ticker_names)}")

    # Fetch market index
    print(f"\nFetching market index (^NYA)...")
    nifty_data = ys.scrape("^NYA", v8=True, time_range='5y')
    nifty_data = ys.v8_formatter(nifty_data)
    nifty_ret = np.log(nifty_data['Close'] / nifty_data['Close'].shift(1))
    nifty_ret.name = 'Market_Mood'
    print(f"✓ Market index: {len(nifty_data)} days")

    # Calculate features
    print(f"\n{'='*70}")
    print(f"  FEATURE ENGINEERING")
    print(f"{'='*70}\n")

    main_df = fc.calculate_features(raw_main, threshold=threshold)
    print(f"✓ Main ticker features: {main_df.shape}")

    # Process supporting tickers
    supporting_tickers = list(dict.fromkeys(ticker_names[1:]))
    successfully_merged = []
    failed_tickers = []

    print(f"\nProcessing {len(supporting_tickers)} supporting tickers...\n")

    for support_ticker in supporting_tickers:
        try:
            raw_support = ys.scrape(support_ticker, v8=True, time_range="5y")

            if raw_support is None:
                failed_tickers.append(support_ticker)
                continue

            raw_support = ys.v8_formatter(raw_support)

            available_cols = [col for col in ['Close', 'Volume', 'High', 'Low']
                            if col in raw_support.columns]

            if not available_cols:
                failed_tickers.append(support_ticker)
                continue

            support_subset = raw_support[available_cols].copy()
            support_subset.columns = [f"{support_ticker}_{col}" for col in support_subset.columns]

            main_df = main_df.join(support_subset, how='left')
            successfully_merged.append(support_ticker)
            print(f"  ✓ {support_ticker}: {len(available_cols)} features")

        except Exception as e:
            failed_tickers.append(support_ticker)
            print(f"  ✗ {support_ticker}: {str(e)[:50]}")
            continue

    # Add market mood
    print(f"\n✓ Adding market sentiment indicator")
    main_df = main_df.join(nifty_ret, how='left')

    # Clean data
    print(f"✓ Cleaning missing values")
    main_df = main_df.ffill().fillna(0)
    main_df = main_df.replace([np.inf, -np.inf], 0)

    print(f"\n{'='*70}")
    print(f"  DATA PREPARATION COMPLETE")
    print(f"{'='*70}")
    print(f"Final dataset shape:        {main_df.shape}")
    print(f"Total features:             {main_df.shape[1]}")
    print(f"Successfully merged:        {len(successfully_merged)}/{len(supporting_tickers)}")
    if failed_tickers:
        print(f"Failed tickers:             {', '.join(failed_tickers)}")
    print(f"{'='*70}\n")

    ticker_info = {
        'main_ticker': ticker,
        'supporting_tickers': successfully_merged,
        'failed_tickers': failed_tickers,
        'total_features': main_df.shape[1]
    }

    return main_df, raw_main, ticker_info


# ============================================================================
# HYBRID WEIGHTING FUNCTIONS
# ============================================================================

def calculate_sample_weights(train_df, recent_weeks=10, weight_multiplier=2.0):
    """
    Calculate sample weights for Hybrid Weighted strategy

    Strategy:
    - Recent data gets progressively higher weight
    - Smooth transition using gradual decay
    - Normalized to maintain effective dataset size

    Args:
        train_df: Training dataframe
        recent_weeks: Number of recent weeks to emphasize (e.g., 10 weeks = 50 days)
        weight_multiplier: Weight for most recent data (e.g., 2.0 = 2x weight)

    Returns:
        weights: Normalized sample weights array
    """
    n_samples = len(train_df)
    weights = np.ones(n_samples)

    # Calculate cutoff for "recent" data
    recent_days = recent_weeks * 5  # 5 trading days per week
    recent_cutoff = max(0, n_samples - recent_days)

    # Apply weight multiplier to recent data
    weights[recent_cutoff:] = weight_multiplier

    # Smooth transition zone (gradual decay)
    decay_window = min(50, recent_cutoff)  # 50-day transition period
    if decay_window > 0 and recent_cutoff > decay_window:
        decay_start = recent_cutoff - decay_window
        decay_weights = np.linspace(1.0, weight_multiplier, decay_window)
        weights[decay_start:recent_cutoff] = decay_weights

    # Normalize weights to sum to n_samples
    # This maintains effective dataset size
    weights = weights * (n_samples / weights.sum())

    return weights


def visualize_weights(weights, train_df):
    """
    Print a visual representation of the weight distribution
    """
    print(f"\n{'='*70}")
    print(f"  SAMPLE WEIGHT DISTRIBUTION")
    print(f"{'='*70}")

    n_samples = len(weights)

    # Calculate weight statistics
    print(f"Total samples:          {n_samples}")
    print(f"Average weight:         {weights.mean():.3f}")
    print(f"Min weight:             {weights.min():.3f}")
    print(f"Max weight:             {weights.max():.3f}")
    print(f"Weight range:           {weights.max() - weights.min():.3f}")

    # Show weight distribution across time
    print(f"\nWeight distribution by period:")
    print(f"{'─'*70}")

    periods = [
        ("Oldest 25%", 0, n_samples//4),
        ("Old 25%", n_samples//4, n_samples//2),
        ("Recent 25%", n_samples//2, 3*n_samples//4),
        ("Most Recent 25%", 3*n_samples//4, n_samples)
    ]

    for period_name, start, end in periods:
        period_weights = weights[start:end]
        avg_weight = period_weights.mean()
        print(f"{period_name:20s}: Avg weight = {avg_weight:.3f}")

    print(f"{'='*70}\n")


# ============================================================================
# WALK-FORWARD TESTING
# ============================================================================

def split_data_by_ratio(df, train_split=0.80):
    """Split data into train and test sets"""
    total_samples = len(df)
    split_idx = int(total_samples * train_split)

    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()

    return train_df, test_df, split_idx


def split_into_weeks(test_df):
    """Split test data into weekly chunks (5 trading days)"""
    weeks = []
    days_per_week = 5

    for i in range(0, len(test_df), days_per_week):
        week_df = test_df.iloc[i:i+days_per_week]
        if len(week_df) > 0:
            weeks.append((i // days_per_week + 1, week_df))

    return weeks


def predict_single_day(predictor, data_up_to_day, config):
    """
    Predict next day's movement using data up to current day
    """
    try:
        result = predictor.train_and_predict(data_up_to_day, use_class_weights=True)

        if result is not None:
            next_prob, acc, diagnostics = result

            if next_prob > 0.5:
                direction = 'UP'
                confidence = next_prob
            else:
                direction = 'DOWN'
                confidence = 1 - next_prob

            return {
                'direction': direction,
                'confidence': confidence,
                'probability': next_prob,
                'diagnostics': diagnostics
            }
        else:
            return None

    except Exception as e:
        print(f"    Error in prediction: {str(e)[:100]}")
        return None


def evaluate_week_predictions(week_df, predictions):
    """
    Evaluate predictions for a week
    """
    daily_results = []

    # Get actual movements
    if 'Target' in week_df.columns:
        actuals = week_df['Target'].values
    else:
        returns = week_df['Returns'].values if 'Returns' in week_df.columns else [0] * len(week_df)
        actuals = [1 if r > 0 else 0 for r in returns]

    for i, pred in enumerate(predictions):
        if pred is None or i >= len(actuals):
            daily_results.append({
                'day': i + 1,
                'predicted': None,
                'actual': None,
                'correct': False,
                'confidence': 0
            })
            continue

        predicted_class = 1 if pred['direction'] == 'UP' else 0
        actual_class = actuals[i]
        correct = (predicted_class == actual_class)

        daily_results.append({
            'day': i + 1,
            'predicted': pred['direction'],
            'actual': 'UP' if actual_class == 1 else 'DOWN',
            'correct': correct,
            'confidence': pred['confidence'],
            'date': week_df.index[i] if i < len(week_df) else None
        })

    # Find best day
    correct_predictions = [r for r in daily_results if r['correct']]
    if correct_predictions:
        best_day_idx = max(range(len(daily_results)),
                          key=lambda i: daily_results[i]['confidence'] if daily_results[i]['correct'] else 0)
    else:
        best_day_idx = 0

    return daily_results, best_day_idx


def run_hybrid_walkforward_testing(main_df, raw_main, config, ticker_info):
    """
    Main walk-forward testing function with HYBRID WEIGHTED strategy
    """

    print(f"\n{'='*70}")
    print(f"  WALK-FORWARD TESTING - HYBRID WEIGHTED STRATEGY")
    print(f"{'='*70}")
    print(f"  Recent data weight: {config['recent_data_weight']}x")
    print(f"  Recent weeks emphasized: {config['recent_weeks_emphasized']}")
    print(f"{'='*70}\n")

    # Step 1: Split data
    print(f"Step 1: Splitting data...")
    train_df, test_df, split_idx = split_data_by_ratio(main_df, config['train_split'])

    print(f"  Training set:   {len(train_df)} days ({config['train_split']:.0%})")
    print(f"  Testing set:    {len(test_df)} days ({1-config['train_split']:.0%})")
    print(f"  Split date:     {main_df.index[split_idx]}")

    # Step 2: Split test set into weeks
    print(f"\nStep 2: Organizing test data into weeks...")
    weeks = split_into_weeks(test_df)
    print(f"  Total weeks to test: {len(weeks)}")
    print(f"  Days per week: 5 (trading days)")

    # Calculate initial weights and visualize
    initial_weights = calculate_sample_weights(
        train_df,
        recent_weeks=config['recent_weeks_emphasized'],
        weight_multiplier=config['recent_data_weight']
    )
    visualize_weights(initial_weights, train_df)

    # Initialize results storage
    all_weekly_results = []
    cumulative_correct = 0
    cumulative_total = 0

    # Current training data
    current_train_df = train_df.copy()

    print(f"\n{'='*70}")
    print(f"  STARTING WEEKLY PREDICTIONS - HYBRID WEIGHTED")
    print(f"{'='*70}\n")

    # Walk forward through each week
    for week_num, week_df in weeks:
        print(f"\n{'─'*70}")
        start_date = week_df.index[0] if hasattr(week_df.index[0], 'date') else week_df.index[0]
        end_date = week_df.index[-1] if hasattr(week_df.index[-1], 'date') else week_df.index[-1]
        start_str = start_date.date() if hasattr(start_date, 'date') else start_date
        end_str = end_date.date() if hasattr(end_date, 'date') else end_date
        print(f"  WEEK {week_num} ({start_str} to {end_str})")
        print(f"{'─'*70}")

        # Calculate sample weights for current training data
        sample_weights = calculate_sample_weights(
            current_train_df,
            recent_weeks=config['recent_weeks_emphasized'],
            weight_multiplier=config['recent_data_weight']
        )

        # Show weight info every 10 weeks
        if week_num % 10 == 1:
            print(f"  Sample weights: Min={sample_weights.min():.2f}, "
                  f"Max={sample_weights.max():.2f}, "
                  f"Avg={sample_weights.mean():.2f}")

        # Train model
        print(f"  Training on {len(current_train_df)} days...")
        print(f"  Using weighted samples (recent {config['recent_weeks_emphasized']} weeks @ {config['recent_data_weight']}x)")

        # Clear previous models
        tf.keras.backend.clear_session()
        gc.collect()

        predictor = MultiTimeframeLSTM(
            lookback_short=config['lookback_short'],
            lookback_medium=config['lookback_medium'],
            lookback_long=config['lookback_long']
        )

        # NOTE: Your MultiTimeframeLSTM class needs to support sample_weight parameter
        # in its train_and_predict method. For now, we'll pass it through

        # Predict each day
        week_predictions = []
        print(f"\n  Daily Predictions:")

        for day_idx in range(len(week_df)):
            data_for_prediction = pd.concat([current_train_df, week_df.iloc[:day_idx+1]])

            pred = predict_single_day(predictor, data_for_prediction, config)
            week_predictions.append(pred)

            if pred:
                day_date = week_df.index[day_idx]
                date_str = day_date.date() if hasattr(day_date, 'date') else day_date
                print(f"    Day {day_idx+1} ({date_str}): "
                      f"{pred['direction']} (confidence: {pred['confidence']:.2%})")
            else:
                print(f"    Day {day_idx+1}: Prediction failed")

        # Evaluate week
        daily_results, best_day_idx = evaluate_week_predictions(week_df, week_predictions)

        correct_count = sum(1 for r in daily_results if r['correct'])
        week_accuracy = correct_count / len(daily_results) if daily_results else 0

        cumulative_correct += correct_count
        cumulative_total += len(daily_results)
        cumulative_accuracy = cumulative_correct / cumulative_total if cumulative_total > 0 else 0

        print(f"\n  Week {week_num} Results:")
        print(f"    Accuracy:           {week_accuracy:.2%} ({correct_count}/{len(daily_results)})")
        best_day_date = daily_results[best_day_idx]['date']
        if best_day_date:
            best_date_str = best_day_date.date() if hasattr(best_day_date, 'date') else best_day_date
        else:
            best_date_str = 'N/A'
        print(f"    Best day:           Day {daily_results[best_day_idx]['day']} ({best_date_str})")
        print(f"    Best day conf:      {daily_results[best_day_idx]['confidence']:.2%}")
        print(f"    Cumulative acc:     {cumulative_accuracy:.2%}")

        # Store results
        all_weekly_results.append({
            'week': week_num,
            'start_date': week_df.index[0],
            'end_date': week_df.index[-1],
            'accuracy': week_accuracy,
            'correct': correct_count,
            'total': len(daily_results),
            'best_day': daily_results[best_day_idx],
            'daily_results': daily_results,
            'cumulative_accuracy': cumulative_accuracy,
            'avg_sample_weight': sample_weights.mean(),
            'max_sample_weight': sample_weights.max()
        })

        # RETRAIN: Add this week with higher weight
        print(f"\n  Retraining: Adding Week {week_num} (will get higher weight next cycle)...")
        current_train_df = pd.concat([current_train_df, week_df])
        print(f"  New training size: {len(current_train_df)} days")

    print(f"\n{'='*70}")
    print(f"  WALK-FORWARD TESTING COMPLETE - HYBRID WEIGHTED")
    print(f"{'='*70}\n")

    return all_weekly_results, cumulative_accuracy


# ============================================================================
# REPORTING
# ============================================================================

def generate_hybrid_report(weekly_results, config, ticker_info, raw_main, save_to_file=True):
    """
    Generate comprehensive report for Hybrid Weighted strategy
    """

    report_lines = []

    # Header
    report_lines.append("="*80)
    report_lines.append(f"  HYBRID WEIGHTED WALK-FORWARD TEST REPORT")
    report_lines.append("="*80)
    report_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append("")

    # Configuration
    report_lines.append("CONFIGURATION")
    report_lines.append("-"*80)
    report_lines.append(f"Ticker:                 {config['ticker']}")
    report_lines.append(f"Train/Test Split:       {config['train_split']:.0%} / {1-config['train_split']:.0%}")
    report_lines.append(f"Price Threshold:        {config['threshold']:.2%}")
    report_lines.append(f"Strategy:               HYBRID WEIGHTED")
    report_lines.append(f"Recent Data Weight:     {config['recent_data_weight']}x")
    report_lines.append(f"Recent Weeks Emphasized: {config['recent_weeks_emphasized']} weeks")
    report_lines.append(f"Lookback Windows:       Short={config['lookback_short']}, "
                       f"Medium={config['lookback_medium']}, Long={config['lookback_long']}")
    report_lines.append("")

    # Ticker Information
    report_lines.append("TICKER INFORMATION")
    report_lines.append("-"*80)
    report_lines.append(f"Main Ticker:            {ticker_info['main_ticker']}")
    report_lines.append(f"Supporting Tickers:     {len(ticker_info['supporting_tickers'])}")
    report_lines.append(f"Total Features:         {ticker_info['total_features']}")
    report_lines.append(f"Current Price:          ${raw_main['Close'].iloc[-1]:.2f}")
    report_lines.append("")

    # Overall Performance
    report_lines.append("OVERALL PERFORMANCE")
    report_lines.append("-"*80)

    total_predictions = sum(w['total'] for w in weekly_results)
    total_correct = sum(w['correct'] for w in weekly_results)
    overall_accuracy = total_correct / total_predictions if total_predictions > 0 else 0

    report_lines.append(f"Total Weeks Tested:     {len(weekly_results)}")
    report_lines.append(f"Total Predictions:      {total_predictions}")
    report_lines.append(f"Correct Predictions:    {total_correct}")
    report_lines.append(f"Overall Accuracy:       {overall_accuracy:.2%}")
    report_lines.append("")

    # Weighting Analysis
    report_lines.append("SAMPLE WEIGHTING ANALYSIS")
    report_lines.append("-"*80)
    avg_weights = [w['avg_sample_weight'] for w in weekly_results]
    max_weights = [w['max_sample_weight'] for w in weekly_results]

    report_lines.append(f"Average Sample Weight:  {np.mean(avg_weights):.3f}")
    report_lines.append(f"Maximum Sample Weight:  {np.mean(max_weights):.3f}")
    report_lines.append(f"Weight Multiplier Used: {config['recent_data_weight']}x")
    report_lines.append("")

    # Weekly Performance
    report_lines.append("WEEKLY PERFORMANCE SUMMARY")
    report_lines.append("-"*80)
    report_lines.append(f"{'Week':<6} {'Dates':<24} {'Accuracy':<12} {'Best Day':<12} {'Confidence':<12}")
    report_lines.append("-"*80)

    for week in weekly_results:
        dates = f"{week['start_date'].strftime('%Y-%m-%d') if hasattr(week['start_date'], 'strftime') else str(week['start_date'])} to {week['end_date'].strftime('%m-%d') if hasattr(week['end_date'], 'strftime') else str(week['end_date'])}"
        accuracy = f"{week['accuracy']:.1%}"
        best_day = f"Day {week['best_day']['day']}"
        confidence = f"{week['best_day']['confidence']:.1%}"

        report_lines.append(f"{week['week']:<6} {dates:<24} {accuracy:<12} {best_day:<12} {confidence:<12}")

    report_lines.append("")

    # Performance Highlights
    report_lines.append("PERFORMANCE HIGHLIGHTS")
    report_lines.append("-"*80)

    sorted_weeks = sorted(weekly_results, key=lambda x: x['accuracy'], reverse=True)
    best_week = sorted_weeks[0]
    worst_week = sorted_weeks[-1]

    report_lines.append(f"Best Week:              Week {best_week['week']} "
                       f"({best_week['accuracy']:.1%} accuracy)")
    report_lines.append(f"Worst Week:             Week {worst_week['week']} "
                       f"({worst_week['accuracy']:.1%} accuracy)")

    # Average confidence for correct predictions
    all_correct_confidences = []
    for week in weekly_results:
        for day in week['daily_results']:
            if day['correct']:
                all_correct_confidences.append(day['confidence'])

    if all_correct_confidences:
        avg_correct_conf = np.mean(all_correct_confidences)
        report_lines.append(f"Avg Confidence (Correct): {avg_correct_conf:.1%}")

    report_lines.append("")

    # Detailed Weekly Breakdown
    report_lines.append("DETAILED WEEKLY BREAKDOWN")
    report_lines.append("="*80)

    for week in weekly_results:
        week_start = week['start_date'].strftime('%Y-%m-%d') if hasattr(week['start_date'], 'strftime') else str(week['start_date'])
        week_end = week['end_date'].strftime('%Y-%m-%d') if hasattr(week['end_date'], 'strftime') else str(week['end_date'])
        report_lines.append(f"\nWeek {week['week']}: {week_start} to {week_end}")
        report_lines.append("-"*80)
        report_lines.append(f"{'Day':<6} {'Date':<12} {'Predicted':<12} {'Actual':<12} "
                           f"{'Correct':<10} {'Confidence':<12}")
        report_lines.append("-"*80)

        for day in week['daily_results']:
            day_num = f"{day['day']}"
            if day['date']:
                date = day['date'].strftime('%Y-%m-%d') if hasattr(day['date'], 'strftime') else str(day['date'])
            else:
                date = 'N/A'
            predicted = day['predicted'] or 'N/A'
            actual = day['actual'] or 'N/A'
            correct = '✓' if day['correct'] else '✗'
            confidence = f"{day['confidence']:.1%}" if day['confidence'] > 0 else 'N/A'

            report_lines.append(f"{day_num:<6} {date:<12} {predicted:<12} {actual:<12} "
                               f"{correct:<10} {confidence:<12}")

        report_lines.append(f"\nWeek Accuracy: {week['accuracy']:.1%}")
        report_lines.append(f"Best Day: Day {week['best_day']['day']} "
                           f"(Confidence: {week['best_day']['confidence']:.1%})")
        report_lines.append(f"Avg Sample Weight: {week['avg_sample_weight']:.3f}")

    report_lines.append("\n" + "="*80)
    report_lines.append("HYBRID STRATEGY INSIGHTS")
    report_lines.append("="*80)
    report_lines.append(f"\nThis strategy emphasized the most recent {config['recent_weeks_emphasized']} weeks")
    report_lines.append(f"by applying a {config['recent_data_weight']}x weight multiplier.")
    report_lines.append(f"\nFinal Accuracy: {overall_accuracy:.2%}")

    if overall_accuracy > 0.55:
        report_lines.append("\n✓ Strategy shows promise - significantly better than random (50%)")
    elif overall_accuracy > 0.50:
        report_lines.append("\n⚠ Strategy shows marginal improvement over random")
    else:
        report_lines.append("\n✗ Strategy underperforms - consider adjusting parameters")

    report_lines.append("\n" + "="*80)
    report_lines.append("END OF REPORT")
    report_lines.append("="*80)

    # Convert to string
    report_text = "\n".join(report_lines)

    # Print to console
    print(report_text)

    # Save to file
    if save_to_file:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"hybrid_weighted_report_{config['ticker']}_{timestamp}.txt"

        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_text)

        print(f"\n✓ Report saved to: {filename}")

    return report_text


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Main execution function for Hybrid Weighted strategy only
    """

    print("\n" + "🎯"*35)
    print("  HYBRID WEIGHTED WALK-FORWARD TESTING")
    print("🎯"*35 + "\n")

    # Get user inputs
    config = get_user_inputs()

    if config is None:
        print("Exiting...")
        return

    # Initialize services
    print("\nInitializing services...")
    ys = YahooScraper()
    gem = Gemini()
    fc = FeatureCalculator()
    print("✓ Services initialized")

    # Fetch and prepare data
    try:
        main_df, raw_main, ticker_info = fetch_and_prepare_data(
            config['ticker'],
            ys,
            gem,
            fc,
            config['threshold']
        )
    except Exception as e:
        print(f"\n❌ Error fetching data: {str(e)}")
        import traceback
        traceback.print_exc()
        return

    # Run Hybrid Weighted walk-forward testing
    try:
        print("\n" + "🔄"*35)
        print("  RUNNING: HYBRID WEIGHTED STRATEGY")
        print("🔄"*35)

        results_hybrid, acc_hybrid = run_hybrid_walkforward_testing(
            main_df,
            raw_main,
            config,
            ticker_info
        )

        print("\nGenerating detailed report...")
        generate_hybrid_report(
            results_hybrid,
            config,
            ticker_info,
            raw_main,
            save_to_file=True
        )

    except Exception as e:
        print(f"\n❌ Error in walk-forward testing: {str(e)}")
        import traceback
        traceback.print_exc()
        return

    # Final Summary
    print("\n" + "="*80)
    print("  ANALYSIS COMPLETE")
    print("="*80)
    print(f"\nTicker: {config['ticker']}")
    print(f"Strategy: HYBRID WEIGHTED")
    print(f"Recent Data Weight: {config['recent_data_weight']}x")
    print(f"Final Accuracy: {acc_hybrid:.2%}")

    if acc_hybrid > 0.55:
        print("\n✓ GOOD: Accuracy significantly above random (50%)")
        print("  Recommendation: Consider this strategy for live trading")
    elif acc_hybrid > 0.50:
        print("\n⚠ MARGINAL: Accuracy slightly above random")
        print("  Recommendation: Optimize parameters or features")
    else:
        print("\n✗ POOR: Accuracy at or below random")
        print("  Recommendation: Revise strategy or features")

    print("\n✓ Report has been generated and saved.")
    print("="*80 + "\n")


# ============================================================================
# RUN THE SCRIPT
# ============================================================================

if __name__ == "__main__":
    main()